# Step 2: Feature Extraction & Processing
## Paper: Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks


Paper: Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks

Author: Kenny Ching

Affiliation: School of Business, University of Auckland

Date: March 2026

Description:
This notebook parses the raw JSON replay telemetry extracted in Step 1. It reconstructs the continuous economic timeline of every match, evaluating combat engagements from both team perspectives to generate the unified, event-level dataset required for our stratified econometric analysis.

Key Calculations:

    Tactical Shock Classification: Scans the Source 2 combat log to identify teamfights, classifying them as either Negative Shocks (net unit deficit) or Positive Shocks (net unit surplus) for the focus team.

    Macroeconomic Trajectories (Strategic Yield, ΔGt​): Calculates the cumulative change in Net Worth Advantage across four discrete temporal horizons (t∈{30,60,120,180} seconds) strictly following the conclusion of the combat event.

    State-Space Variables: Extracts the exact game clock (fight_start) and economic baseline (current_gold_lead) at the moment of the shock to enable Nearest-Neighbor (k=5) state-space matching in Step 3.

Input:

    Raw JSON match files from data/raw/.

Output:

    data/processed/tf_events_master_unified.csv (The master econometric dataset).

In [ ]:
import os
import glob
import json
import pandas as pd
from tqdm import tqdm
from google.colab import drive

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

DATA_DIR = "/content/drive/My Drive/OpenAI_Data/data"
SAVE_PATH = os.path.join(DATA_DIR, "tf_events_master_unified.csv")

OPENAI_IDS = [4693543125, 4693543123, 4693543117, 4080601137, 4080420268, 4080316101]
TI_LEAGUES = {9870: 'TI8', 10749: 'TI9'}
HORIZONS = [30, 60, 120, 180]

def get_gold_at_time(gold_adv_list, time_sec):
    if not gold_adv_list: return 0
    idx = int(time_sec / 60)
    if idx < len(gold_adv_list): return gold_adv_list[idx]
    return gold_adv_list[-1]

# ==========================================
# 2. DATA EXTRACTION
# ==========================================
def parse_unified_match(file_path):
    try:
        with open(file_path, 'r') as f: data = json.load(f)
    except: return []

    match_id = data.get('match_id')
    league_id = data.get('leagueid', 0)
    is_openai_match = (match_id in OPENAI_IDS)

    if not is_openai_match and league_id not in TI_LEAGUES: return []

    radiant_win = data.get('radiant_win')
    gold_adv = data.get('radiant_gold_adv', [])
    if not gold_adv: return []

    events = []
    for tf in data.get('teamfights', []):
        start = tf['start']

        r_deaths = sum(p.get('deaths', 0) for i, p in enumerate(tf.get('players', [])) if i < 5)
        d_deaths = sum(p.get('deaths', 0) for i, p in enumerate(tf.get('players', [])) if i >= 5)
        if r_deaths == 0 and d_deaths == 0: continue

        for team in ['radiant', 'dire']:
            my_net_kills = (d_deaths - r_deaths) if team == 'radiant' else (r_deaths - d_deaths)

            # Classify Shock Type
            if my_net_kills < 0: shock_type = 'negative' # Resilience
            elif my_net_kills > 0: shock_type = 'positive' # Exploitation
            else: continue # Ignore neutral trades

            # Identify AI
            is_ai = False
            if is_openai_match:
                t_name = data.get(f'{team}_team', {}).get('name', '').lower()
                if 'openai' in t_name: is_ai = True
                elif 'openai' not in data.get('radiant_team', {}).get('name', '').lower() and \
                     'openai' not in data.get('dire_team', {}).get('name', '').lower():
                     if team == ('radiant' if radiant_win else 'dire'): is_ai = True

            did_win = (radiant_win and team == 'radiant') or (not radiant_win and team == 'dire')

            base_g = get_gold_at_time(gold_adv, start)
            current_lead = base_g if team == 'radiant' else -base_g

            yields = {}
            for t in HORIZONS:
                change = get_gold_at_time(gold_adv, start + t) - base_g
                yields[f'gadv_d{t}'] = change if team == 'radiant' else -change

            events.append({
                'match_id': match_id,
                'shock_type': shock_type,
                'is_ai': int(is_ai),
                'did_win': int(did_win),
                'fight_start': start,
                'current_gold_lead': current_lead,
                **yields
            })
    return events

if __name__ == "__main__":
    print("Extracting Unified Data...")
    files = glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True)
    all_events = []
    for f in tqdm(files): all_events.extend(parse_unified_match(f))
    df = pd.DataFrame(all_events)
    df.to_csv(SAVE_PATH, index=False)
    print(f"\n✅ SUCCESS: Extracted {len(df)} discrete tactical shocks to {SAVE_PATH}.")